In [ ]:
%%capture
%pip install evaluate

In [9]:
# Les importations
from transformers import get_linear_schedule_with_warmup, AutoTokenizer,TrainerCallback,AutoModelForCausalLM, DataCollatorForSeq2Seq,AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer,EarlyStoppingCallback
from datasets import load_dataset,Dataset, DatasetDict
import evaluate
import numpy as np
from torch.cuda.amp import autocast, GradScaler
import torch
import tensorflow as tf
import matplotlib.pyplot as plt

### Chargement du dataset

In [14]:
def dataset_loader(path) :
    with open(path, "r") as file:
        data = file.readlines()

    data_clean = [line for line in data if line.strip()]

    return data_clean


In [15]:
data_arb = dataset_loader("/kaggle/input/arb-fr-2/arabe_tchad3.txt")
data_fr = dataset_loader("/kaggle/input/arb-fr-2/V_francais_3.txt")

In [16]:
print(type(data_arb))

<class 'list'>


In [ ]:
# for trying 
data_arb = data_arb[:3000] 
data_fr = data_fr[:3000]

### Diviosion du dataset en train, test et validation

In [19]:
# Création du dictionnaire
dataset_dict = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb, data_fr)]}

# Création du dataset
dataset = Dataset.from_dict(dataset_dict)
# On va diviser notre dataset en deux parties
 #train_dataset =  DatasetDict({"train":test['train']})

train_test_dataset = dataset.train_test_split(test_size=0.2, seed= 42, shuffle= True)

test_val_dataset = train_test_dataset['test'].train_test_split(test_size= 0.5, seed= 42)

train_test_val_dataset = DatasetDict ({
    'train' : train_test_dataset['train'],
    'val'   : test_val_dataset['train'],
    'test'  : test_val_dataset['test']
})
train_test_val_dataset.save_to_disk("./Data")


Saving the dataset (0/1 shards):   0%|          | 0/800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]